In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')

# --- ПУТИ ---
BASE_DIR = Path(r"C:\Users\egor0\ProjectsVSCode\ML_Practice")
DATA_DIR = BASE_DIR / "data"
PARQUET_DIR = DATA_DIR / "parquet"          # Здесь лежат исходные раздробленные файлы
CLEAN_DIR = DATA_DIR / "clean"              # Здесь будут лежать очищенные train/val

# Создаем папку, если её нет
CLEAN_DIR.mkdir(exist_ok=True)

# --- НАСТРОЙКИ ОТОБРАЖЕНИЯ ---
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# --- КОНСТАНТЫ ДЛЯ ЧТЕНИЯ/ЗАПИСИ ---
CHUNK_SIZE = 10_000                         # Размер чанка при чтении CSV
ENGINE = 'fastparquet'                      # Движок для Parquet (fastparquet или pyarrow)
RANDOM_STATE = 42                           # Фиксируем seed для воспроизводимости

features_to_keep_final = [] 
cat_cols_present = [
    'ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo', 
    'id_12', 'id_15', 'id_16', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 
    'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 
    'id_36', 'id_37', 'id_38'
]


In [2]:
import gc


first_file = sorted(PARQUET_DIR.glob('train_trans_*.parquet'))[0]
df_sample = pd.read_parquet(first_file, engine=ENGINE)


all_cols = df_sample.columns.tolist()
service_cols = ['TransactionID', 'isFraud']

features_to_keep_final = [
    c for c in all_cols 
    if c not in service_cols and c not in cat_cols_present
]

del df_sample
gc.collect()

print(f"Найдено числовых признаков: {len(features_to_keep_final)}")
print(f"Найдено категориальных признаков: {len(cat_cols_present)}")
print(f"Всего признаков для модели: {len(features_to_keep_final) + len(cat_cols_present)}")

assert len(features_to_keep_final) > 0, "Список числовых признаков пуст!"
assert len(set(features_to_keep_final) & set(cat_cols_present)) == 0, "Числовые и категориальные пересекаются!"

Найдено числовых признаков: 378
Найдено категориальных признаков: 37
Всего признаков для модели: 415


In [4]:
import time
import gc
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

train_files = sorted(glob.glob(str(PARQUET_DIR / 'train_trans_*.parquet')))
id_target_chunks = []
for f in train_files:
    chunk = pd.read_parquet(f, columns=['TransactionID', 'isFraud'], engine=ENGINE)
    id_target_chunks.append(chunk)
    del chunk

id_target_df = pd.concat(id_target_chunks, ignore_index=True)
del id_target_chunks
gc.collect()

print(f"Всего транзакций: {len(id_target_df):,}")

train_idx, val_idx = train_test_split(
    id_target_df.index, 
    test_size=0.2, 
    stratify=id_target_df['isFraud'], 
    random_state=RANDOM_STATE
)

train_ids = set(id_target_df.loc[train_idx, 'TransactionID'])
val_ids = set(id_target_df.loc[val_idx, 'TransactionID'])

print(f"-> train: {len(train_ids):,}, val: {len(val_ids):,}")

del id_target_df, train_idx, val_idx
gc.collect()

train_chunks = []
val_chunks = []

for i, f in enumerate(train_files):
    chunk = pd.read_parquet(f, engine=ENGINE)
    
    # Оставляем только нужные признаки
    cols_to_use = [c for c in features_to_keep_final + cat_cols_present + ['TransactionID', 'isFraud'] if c in chunk.columns]
    chunk = chunk[cols_to_use].copy()
    
    mask_train = chunk['TransactionID'].isin(train_ids)
    mask_val = chunk['TransactionID'].isin(val_ids)
    
    if mask_train.any():
        train_chunks.append(chunk[mask_train].copy())
    if mask_val.any():
        val_chunks.append(chunk[mask_val].copy())
        
    del chunk
    if i % 10 == 0:
        print(f"  Обработано файлов: {i+1}/{len(train_files)}")
        gc.collect()

train_df = pd.concat(train_chunks, ignore_index=True)
val_df = pd.concat(val_chunks, ignore_index=True)
del train_chunks, val_chunks
gc.collect()

print("\nУдаление выбросов в TransactionAmt (IQR)...")
col = 'TransactionAmt'

Q1 = train_df[col].quantile(0.25)
Q3 = train_df[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"  Границы: [{lower_bound:.2f}, {upper_bound:.2f}]")

train_df_clean = train_df[(train_df[col] >= lower_bound) & (train_df[col] <= upper_bound)].copy()

dropped_count = len(train_df) - len(train_df_clean)
dropped_pct = dropped_count / len(train_df) * 100
print(f"Train после удаления выбросов: {len(train_df_clean):,} (удалено {dropped_count:,} строк, {dropped_pct:.2f}%)")

fraud_left = train_df_clean['isFraud'].sum()
print(f"Мошенничеств осталось: {fraud_left:,} ({fraud_left/len(train_df_clean)*100:.2f}%)")

del train_df
gc.collect()


num_cols_present = [c for c in features_to_keep_final if c in train_df_clean.columns]

for col in num_cols_present:
    median_val = train_df_clean[col].median()
    
    if pd.isna(median_val):
        median_val = 0.0
        
    train_df_clean[col] = train_df_clean[col].fillna(median_val)
    
    if col in val_df.columns:
        val_df[col] = val_df[col].fillna(median_val)

gc.collect()
print("Пропуски заполнены!")


del train_ids, val_ids
gc.collect()


extra_cols = [c for c in ['TransactionDT'] if c in train_df_clean.columns]
cat_cols_present_valid = [c for c in cat_cols_present if c in train_df_clean.columns]

raw_cols = num_cols_present + cat_cols_present_valid + extra_cols + ['isFraud']
final_cols = list(dict.fromkeys(raw_cols))

print(f"Колонок до очистки от дубликатов: {len(raw_cols)}")
print(f"Колонок после очистки: {len(final_cols)}")

train_df_clean[final_cols].to_parquet(
    CLEAN_DIR / 'train_part.parquet', 
    index=False, 
    engine=ENGINE
)
print("  train_part.parquet сохранен")

val_df[final_cols].to_parquet(
    CLEAN_DIR / 'val_part.parquet', 
    index=False, 
    engine=ENGINE
)
print("  val_part.parquet сохранен")

print("\n" + "="*40)


del train_df_clean, val_df
gc.collect()

Всего транзакций: 590,540
-> train: 472,432, val: 118,108
  Обработано файлов: 1/60
  Обработано файлов: 11/60
  Обработано файлов: 21/60
  Обработано файлов: 31/60
  Обработано файлов: 41/60
  Обработано файлов: 51/60

Удаление выбросов в TransactionAmt (IQR)...
  Q1: 43.14, Q3: 125.00, IQR: 81.86
  Границы: [-79.65, 247.79]
Train после удаления выбросов: 419,221 (удалено 53,211 строк, 11.26%)
Мошенничеств осталось: 13,833 (3.30%)
Пропуски заполнены!
Колонок до очистки от дубликатов: 394
Колонок после очистки: 393
  train_part.parquet сохранен
  val_part.parquet сохранен



0